# Modules.

In [1]:
#Import modules.
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd
import numpy as np
import os
import csv
import tensorflow as tf
from matplotlib import pyplot as plt
from keras.models import Sequential
from keras import layers
from keras.optimizers import RMSprop
from scipy.optimize import curve_fit
import scipy as sp

# Upload data and preprocessing.

In [130]:
!unzip Unidos\ contra\ la\ COVID-19.csv.zip

Archive:  Unidos contra la COVID-19.csv(1).zip
replace Unidos contra la COVID-19.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: Unidos contra la COVID-19.csv  


In [2]:
df = pd.read_csv('COVID-19.csv')
#df.head()

In [3]:
df = df.replace({"No sé", "Sí", 360})
df = df.replace({"No sé": "No"})

### Preprocess

In [4]:
# Preprocessing.

## Rename columns with p_number of question.
preguntas = list(df.columns)
preg = ['p_{}'.format(i) for i in range(-1,31)]
df.columns = preg

## Get rid of 'No' answers in p_0.
df = df[df['p_0']!='No']
df.reset_index(inplace=True)
ans = len(df)

## Hand-made corrections (p_12).
df.loc[44, 'p_12'] = 3
df.loc[218, 'p_12'] = 20
df.loc[542, 'p_12'] = 3
df.loc[372, 'p_12'] = 10
df.loc[14, 'p_12'] = 1
df.loc[234, 'p_12'] = 12
df.loc[1838, 'p_12'] = 2
df.loc[2113, 'p_12'] = 3
df.loc[1605, 'p_12'] = 3

In [5]:
print('NaNs in each question:')
for i in range(0,31):
  null_cells = df['p_{}'.format(i)].isnull().sum()
  if null_cells != 0:
    print('p{}:'.format(i), null_cells)

NaNs in each question:
p6: 2191
p10: 2294
p12: 1613
p13: 1613
p15: 2020
p17: 2026
p18: 2026


In [6]:
#Questions with NaNs
print(preguntas[7])
print(preguntas[11])
print(preguntas[13])
print(preguntas[14])
print(preguntas[16])
print(preguntas[18])
print(preguntas[19])

¿Cuál de las siguientes opciones describe mejor el lugar donde vivías anteriormente? (En caso de haber respondido que te has mudado debido a la pandemia)
¿Hace cuánto tiempo?
De los síntomas descritos en la pregunta anterior (fiebre, tos o dificultad para respirar). ¿A cuántas personas conoces con estos síntomas? Sólo escribe EL NÚMERO aproximado (NO TEXTO).
¿Estuviste con alguna de estas personas en los últimos 7 días?
¿En qué modalidad estás laborando actualmente?
En la mayoría de las veces que has usado transporte público, ¿qué tan saturado se encuentra el servicio?
En un viaje de ida y vuelta que hayas efectuado en transporte público, ¿cuál es el tiempo promedio que dura tu viaje?


In [7]:
## Fill NaNs with 0s.
df.fillna(0, inplace=True)
df.p_7 = df.p_7.replace({"1", "2", 306})
df.p_7 = df.p_7.replace({"1": 0})

## Set X as a copy of df.
X = df.copy().loc[:,'p_1':].astype(str)
X.fillna(0, inplace=True)
X = X.values

## Ordinal encoding w/ skl.
oe = OrdinalEncoder()
oe.fit(X)
X_enc = oe.transform (X)

In [8]:
oe.categories_[11]

array(['0', '1', '10', '100', '12', '14', '15', '16', '17', '2', '20',
       '25', '26', '3', '30', '4', '40', '5', '6', '7', '8', '9',
       'Fiebre', 'Tres', 'Uno', 'Yos'], dtype=object)

In [9]:
enc_df = pd.DataFrame(X_enc, columns = preg[2:])
enc_df.head()

,p_1,p_2,p_3,p_4,p_5,p_6,p_7,p_8,p_9,p_10,...,p_21,p_22,p_23,p_24,p_25,p_26,p_27,p_28,p_29,p_30
0,0.0,0.0,3.0,0.0,1.0,0.0,0.0,89.0,0.0,0.0,...,41.0,1.0,33.0,5.0,0.0,0.0,1.0,12.0,15.0,246.0
1,1.0,0.0,3.0,0.0,2.0,0.0,1.0,133.0,0.0,4.0,...,40.0,1.0,29.0,5.0,0.0,0.0,2.0,29.0,16.0,136.0
2,1.0,0.0,6.0,1.0,1.0,2.0,0.0,0.0,0.0,0.0,...,40.0,1.0,6.0,5.0,0.0,0.0,0.0,28.0,8.0,136.0
3,0.0,0.0,6.0,0.0,2.0,0.0,1.0,134.0,0.0,4.0,...,40.0,1.0,37.0,1.0,0.0,0.0,0.0,6.0,4.0,159.0
4,1.0,0.0,6.0,0.0,1.0,0.0,0.0,133.0,0.0,0.0,...,40.0,1.0,58.0,5.0,0.0,0.0,0.0,12.0,24.0,144.0


In [10]:
enc_df.to_csv('encoded.csv')

# Hand encoding.

In [ ]:
#Question 1. (Sex/Gender)
for i in range(ans):
  if df['p_1'][i] == 'Hombre':
    df['p_1'][i] = 0
  elif df['p_1'][i] == 'Mujer':
    df['p_1'][i] = 1
  elif df['p_1'][i] == 'Otro':
    df['p_1'][i] = 2
  else:
    df['p_1'][i] = 3

In [ ]:
#Question 2. (Age range)
for i in range(ans):
  if df['p_2'][i] == 'Entre 18 a 24 años':
    df['p_2'][i] = 0
  elif df['p_2'][i] == 'Entre 25 años a 34 años':
    df['p_2'][i] = 1
  elif df['p_2'][i] == 'Entre 35 años a 44 años':
    df['p_2'][i] = 2
  elif df['p_2'][i] == 'Entre 45 años a 54 años':
    df['p_2'][i] = 3
  elif df['p_2'][i] == 'Entre 55 años a 64 años':
    df['p_2'][i] = 4
  elif df['p_2'][i] == 'Entre 65 años a 74 años':
    df['p_2'][i] = 5
  elif df['p_2'][i] == '75 años o más':
    df['p_2'][i] = 6

In [ ]:
#Question 3. (Where do you live?)
for i in range(ans):
  if df['p_3'][i] == 'Aguascalientes':
    df['p_3'][i] = 0
  elif df['p_3'][i] == 'Baja California Norte':
    df['p_3'][i] = 1
  elif df['p_3'][i] == 'Baja California Sur':
    df['p_3'][i] = 2
  elif df['p_3'][i] == 'Campeche':
    df['p_3'][i] = 3
  elif df['p_3'][i] == 'Ciudad de México':
    df['p_3'][i] = 4
  elif df['p_3'][i] == 'Chihuahua':
    df['p_3'][i] = 5
  elif df['p_3'][i] == 'Chiapas':
    df['p_3'][i] = 6
  elif df['p_3'][i] == 'Coahuila de Zaragoza':
    df['p_3'][i] = 7
  elif df['p_3'][i] == 'Colima':
    df['p_3'][i] = 8
  elif df['p_3'][i] == 'Durango':
    df['p_3'][i] = 9
  elif df['p_3'][i] == 'Guanajuato':
    df['p_3'][i] = 10
  elif df['p_3'][i] == 'Guerrero':
    df['p_3'][i] = 11
  elif df['p_3'][i] == 'Hidalgo':
    df['p_3'][i] = 12
  elif df['p_3'][i] == 'Jalisco':
    df['p_3'][i] = 13
  elif df['p_3'][i] == 'Estado de México':
    df['p_3'][i] = 14
  elif df['p_3'][i] == 'Michoacán de Ocampo':
    df['p_3'][i] = 15
  elif df['p_3'][i] == 'Morelos':
    df['p_3'][i] = 16
  elif df['p_3'][i] == 'Nayarit':
    df['p_3'][i] = 17
  elif df['p_3'][i] == 'Nuevo León':
    df['p_3'][i] = 18
  elif df['p_3'][i] == 'Oaxaca':
    df['p_3'][i] = 19
  elif df['p_3'][i] == 'Puebla':
    df['p_3'][i] = 20
  elif df['p_3'][i] == 'Querétaro':
    df['p_3'][i] = 21
  elif df['p_3'][i] == 'Quintana Roo':
    df['p_3'][i] = 22
  elif df['p_3'][i] == 'San Luis Potosí':
    df['p_3'][i] = 23
  elif df['p_3'][i] == 'Sinaloa':
    df['p_3'][i] = 24
  elif df['p_3'][i] == 'Sonora':
    df['p_3'][i] = 25
  elif df['p_3'][i] == 'Tabasco':
    df['p_3'][i] = 26
  elif df['p_3'][i] == 'Tamaulipas':
    df['p_3'][i] = 27
  elif df['p_3'][i] == 'Tlaxcala':
    df['p_3'][i] = 28
  elif df['p_3'][i] == 'Veracruz':
    df['p_3'][i] = 29
  elif df['p_3'][i] == 'Yucatán':
    df['p_3'][i] = 30
  elif df['p_3'][i] == 'Zacatecas':
    df['p_3'][i] = 31

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  if sys.path[0] == '':
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:64: SettingWithCopyWarning:

In [ ]:
#Question 4. (Did you change your living place?)
for i in range(ans):
  if df['p_4'][i] == 'No':
    df['p_4'][i] = 0
  elif df['p_4'][i] == 'Sí':
    df['p_4'][i] = 1

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  after removing the cwd from sys.path.
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


In [ ]:
#Question 5. (What kind of place is where you live?)
for i in range(ans):
  if df['p_5'][i] == 'Ciudad':
    df['p_5'][i] = 1
  elif df['p_5'][i] == 'Pueblo':
    df['p_5'][i] = 2
  elif df['p_5'][i] == 'Campo':
    df['p_5'][i] = 3
  elif df['p_5'][i] == 'Zona Rural':
    df['p_5'][i] = 4

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  after removing the cwd from sys.path.
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  # Remove the CWD from sys.path while we load stuff.
/usr/local/lib/py

In [ ]:
#Question 6. (Where did you used to live?)
for i in range(ans):
  if df['p_6'][i] == 'Ciudad':
    df['p_6'][i] = 1
  elif df['p_6'][i] == 'Pueblo':
    df['p_6'][i] = 2
  elif df['p_6'][i] == 'Campo':
    df['p_6'][i] = 3
  elif df['p_6'][i] == 'Zona Rural':
    df['p_6'][i] = 4

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  after removing the cwd from sys.path.
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  # Remove the CWD from sys.path while we load stuff.


In [ ]:
#Question 7. (Have you infected with COVID-19?)
for i in range(ans):
  if df['p_7'][i] == 'No':
    df['p_7'][i] = 0
  elif df['p_7'][i] == 'Sí':
    df['p_7'][i] = 1
  elif df['p_7'][i] == 'No sé':
    df['p_7'][i] = 1

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  after removing the cwd from sys.path.
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


In [ ]:
#Question 8. (Did you had any of these symptoms?)
for i in range(ans):
  if df['p_8'][i] == 'No he tenido síntomas':
    df['p_8'][i] = 0
  elif df['p_8'][i] == 'Tos':
    df['p_8'][i] = 1
  elif df['p_8'][i] == 'Congestión nasal':
    df['p_8'][i] = 2
  elif df['p_8'][i] == 'Dificultad para respirar':
    df['p_8'][i] = 3
  elif df['p_8'][i] == 'Pérdida del sentido del gusto o el olfato':
    df['p_8'][i] = 4
  elif df['p_8'][i] == 'Fatiga':
    df['p_8'][i] = 5
  elif df['p_8'][i] == 'Dolor de garganta':
    df['p_8'][i] = 6
  elif df['p_8'][i] == 'Dolor de pecho':
    df['p_8'][i] = 7
  elif df['p_8'][i] == 'Dolor de cabeza':
    df['p_8'][i] = 8
  elif df['p_8'][i] == 'Dolores musculares':
    df['p_8'][i] = 9
  elif df['p_8'][i] == 'Fiebre':
    df['p_8'][i] = 10
  elif df['p_8'][i] == 'Escalofríos':
    df['p_8'][i] = 11

In [ ]:
#Question 9. (Have you done a COVID test?)
for i in range(ans):
  if df['p_9'][i] == 'No':
    df['p_9'][i] = 0
  elif df['p_9'][i] == 'Sí':
    df['p_9'][i] = 1

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  after removing the cwd from sys.path.
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


In [ ]:
#Question 10. (How much time ago did you do the test?)
for i in range(ans):
  if df['p_10'][i] == 'Menos de una semana':
    df['p_10'][i] = 1
  elif df['p_10'][i] == 'Entre una semana y quince días':
    df['p_10'][i] = 2
  elif df['p_10'][i] == 'Entre quince días y un mes':
    df['p_10'][i] = 3
  elif df['p_10'][i] == 'Más de un mes':
    df['p_10'][i] = 4

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  # Remove the CWD from sys.path while we load stuff.
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  after removing the cwd from sys.path.
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/py

In [ ]:
#Question 11. (Do you know someone with symptoms?)
for i in range(ans):
  if df['p_11'][i] == 'No':
    df['p_11'][i] = 0
  elif df['p_11'][i] == 'Sí':
    df['p_11'][i] = 1

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  after removing the cwd from sys.path.
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


In [ ]:
#Question 13. (Did you see some of this persons in the last 7 days?)
for i in range(ans):
  if df['p_13'][i] == 'No':
    df['p_13'][i] = 0
  elif df['p_13'][i] == 'Sí':
    df['p_13'][i] = 1

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  after removing the cwd from sys.path.
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


In [ ]:
#Question 14. (Do you work?)
for i in range(ans):
  if df['p_14'][i] == 'No':
    df['p_14'][i] = 0
  elif df['p_14'][i] == 'Sí':
    df['p_14'][i] = 1

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  after removing the cwd from sys.path.
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


In [ ]:
#Question 15. (Which manner?)
for i in range(ans):
  if df['p_15'][i] == 'Trabajo a distancia':
    df['p_15'][i] = 1
  elif df['p_15'][i] == 'Trabajo presencial':
    df['p_15'][i] = 2
  elif df['p_15'][i] == 'Mixto':
    df['p_15'][i] = 3

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  after removing the cwd from sys.path.
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


In [ ]:
#Question 16. (Do you use public transport frecuently?)
for i in range(ans):
  if df['p_16'][i] == 'No':
    df['p_16'][i] = 0
  elif df['p_16'][i] == 'Sí':
    df['p_16'][i] = 1

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  after removing the cwd from sys.path.
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


In [ ]:
#Question 17. (How much people is there?)
for i in range(ans):
  if df['p_17'][i] == 'Vacío':
    df['p_17'][i] = 1
  elif df['p_17'][i] == 'Semivacío':
    df['p_17'][i] = 2
  elif df['p_17'][i] == 'Semi Saturado':
    df['p_17'][i] = 3
  elif df['p_17'][i] == 'Saturado':
    df['p_17'][i] = 4

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  # Remove the CWD from sys.path while we load stuff.
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launc

In [ ]:
#Question 18. (How much time do you spend in transportation?)
for i in range(ans):
  if df['p_18'][i] == 'Menos de 30 minutos':
    df['p_18'][i] = 1
  elif df['p_18'][i] == 'Entre 30 minutos a hora y media':
    df['p_18'][i] = 2
  elif df['p_18'][i] == 'Entre hora y media a tres horas':
    df['p_18'][i] = 3
  elif df['p_18'][i] == 'Más de tres horas':
    df['p_18'][i] = 4

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  after removing the cwd from sys.path.
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:10: Sett

In [ ]:
#Question 19. (Did you do any of these activities?)
for i in range(ans):
  if df['p_19'][i] == 'Ir a trabajar fuera del lugar en el que estás viviendo actualmente':
    df['p_19'][i] = 0
  elif df['p_19'][i] == 'Ir a un mercado, una tienda de alimentos o farmacia':
    df['p_19'][i] = 1
  elif df['p_19'][i] == 'Ir a un restaurante, cafetería o centro comercial':
    df['p_19'][i] = 2
  elif df['p_19'][i] == 'Pasar tiempo con alguien que no esté viviendo contigo actualmente':
    df['p_19'][i] = 3
  elif df['p_19'][i] == 'Asistir a un evento público con más de 10 personas':
    df['p_19'][i] = 4
  elif df['p_19'][i] == 'Usar transporte público':
    df['p_19'][i] = 5
  elif df['p_19'][i] == 'Ninguna de las anteriores':
    df['p_19'][i] = 5

In [ ]:
#Question 20. (What kind of transportation do you use the most?)
for i in range(ans):
  if df['p_20'][i] == 'Metro, Tren suburbano o Tren ligero':
    df['p_20'][i] = 0
  elif df['p_20'][i] == 'Metrobus, Mexibus o Mexicable':
    df['p_20'][i] = 1
  elif df['p_20'][i] == 'Autobús, camión, combi o trolebús':
    df['p_20'][i] = 2
  elif df['p_20'][i] == 'Taxi, UBER, DiDi':
    df['p_20'][i] = 3
  elif df['p_20'][i] == 'Vehículo propio o prestado (auto, motocicleta)':
    df['p_20'][i] = 4
  elif df['p_20'][i] == 'Bicicleta, patines, patineta u otro':
    df['p_20'][i] = 5

In [ ]:
#Question 21. (Have you done some of these in the last month?)
for i in range(ans):
  if df['p_21'][i] == 'Ir a fiesta, boda o bautizo':
    df['p_21'][i] = 0
  elif df['p_21'][i] == 'Ir a un hospital como voluntario, paciente o visita':
    df['p_21'][i] = 1
  elif df['p_21'][i] == 'Ir a un antro, discoteca o bar':
    df['p_21'][i] = 2
  elif df['p_21'][i] == 'Ir a un concierto, feria, o tianguis concurrido':
    df['p_21'][i] = 3
  elif df['p_21'][i] == 'Pasar tiempo frecuentemente con alguien que no esté viviendo contigo actualmente':
    df['p_21'][i] = 4
  elif df['p_21'][i] == 'Asistir a eventos público con más de 10 personas':
    df['p_21'][i] = 5
  elif df['p_21'][i] == 'Ninguna de las anteriores':
    df['p_21'][i] = 6

In [ ]:
#Question 22. (Do you wash your hand after being out?)
for i in range(ans):
  if df['p_22'][i] == 'No':
    df['p_22'][i] = 0
  elif df['p_22'][i] == 'Sí':
    df['p_22'][i] = 1

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  after removing the cwd from sys.path.


In [ ]:
#Question 23. (Which mask do you use?)
for i in range(ans):
  if df['p_23'][i] == 'Desechable':
    df['p_23'][i] = 0
  elif df['p_23'][i] == 'Neopreno':
    df['p_23'][i] = 1
  elif df['p_23'][i] == 'Tela':
    df['p_23'][i] = 2
  elif df['p_23'][i] == 'Quirúrgico':
    df['p_23'][i] = 3
  elif df['p_23'][i] == 'KN95':
    df['p_23'][i] = 4
  elif df['p_23'][i] == 'Mascarilla con válvula':
    df['p_23'][i] = 5
  elif df['p_23'][i] == 'Otro (Bufanda, paliacate, trapo, etc)':
    df['p_23'][i] = 6
  elif df['p_23'][i] == 'No uso':
    df['p_23'][i] = 7

In [ ]:
#Question 24. (Do you use mask in public places?)
for i in range(ans):
  if df['p_24'][i] == 'Siempre':
    df['p_24'][i] = 0
  elif df['p_24'][i] == 'La mayoría de las veces':
    df['p_24'][i] = 1
  elif df['p_24'][i] == 'A veces':
    df['p_24'][i] = 2
  elif df['p_24'][i] == 'Pocas veces':
    df['p_24'][i] = 3
  elif df['p_24'][i] == 'Nunca':
    df['p_24'][i] = 4
  elif df['p_24'][i] == 'No estuve en ningún lugar público en los últimos 7 días':
    df['p_24'][i] = 5

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  after removing the cwd from sys.path.
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:8: Sett

In [ ]:
#Question 25. (How do you use your mask?)
for i in range(ans):
  if df['p_25'][i] == 'Debajo de la barbilla':
    df['p_25'][i] = 0
  elif df['p_25'][i] == 'Debajo de la nariz':
    df['p_25'][i] = 1
  elif df['p_25'][i] == 'Completamente cubierto':
    df['p_25'][i] = 2

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  after removing the cwd from sys.path.


In [ ]:
#Question 26. (How the people whom you live with use their masks?)
for i in range(ans):
  if df['p_26'][i] == 'Debajo de la barbilla':
    df['p_26'][i] = 0
  elif df['p_26'][i] == 'Debajo de la nariz':
    df['p_26'][i] = 1
  elif df['p_26'][i] == 'Completamente cubierto':
    df['p_26'][i] = 2

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  after removing the cwd from sys.path.


In [ ]:
#Question 27. (How many days do you were with people that don't live with you?)
for i in range(ans):
  if df['p_27'][i] == '0 días':
    df['p_27'][i] = 0
  elif df['p_27'][i] == '1 día':
    df['p_27'][i] = 1
  elif df['p_27'][i] == 'Entre 2 a 4 días':
    df['p_27'][i] = 2
  elif df['p_27'][i] == 'Entre 5 a 7 días':
    df['p_27'][i] = 3

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  after removing the cwd from sys.path.
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:10: Sett

In [ ]:
#Question 28. (How do you get information about the pandemic?)
for i in range(ans):
  if df['p_28'][i] == 'En pláticas con familiares, conocidos o amigos':
    df['p_28'][i] = 0
  elif df['p_28'][i] == 'Por grupos de chat de familiares, amigos, colegas de trabajo':
    df['p_28'][i] = 1
  elif df['p_28'][i] == 'Por medios tradicionales (TV, Radio, Periódicos)':
    df['p_28'][i] = 2
  elif df['p_28'][i] == 'Por redes sociales (Facebook, Twitter, Instagram, Tiktok)':
    df['p_28'][i] = 3
  elif df['p_28'][i] == 'Por instituciones médicas u organismos de especialistas':
    df['p_28'][i] = 4

In [ ]:
#Question 29. (What social media do you consult the most?)
for i in range(ans):
  if df['p_29'][i] == 'Facebook':
    df['p_29'][i] = 0
  elif df['p_29'][i] == 'Twitter':
    df['p_29'][i] = 1
  elif df['p_29'][i] == 'Instagram':
    df['p_29'][i] = 2
  elif df['p_29'][i] == 'Tiktok':
    df['p_29'][i] = 3
  elif df['p_29'][i] == 'YouTube':
    df['p_29'][i] = 4
  elif df['p_29'][i] == 'Ninguna, no tengo redes sociales':
    df['p_29'][i] = 5

In [ ]:
#Question 30. (What kinds of accounts do you follow?)
for i in range(ans):
  if df['p_30'][i] == 'Gobierno Federal, Secretaría de Salud, IMSS, ISSSTE':
    df['p_30'][i] = 0
  elif df['p_30'][i] == 'Organismos internacionales':
    df['p_30'][i] = 1
  elif df['p_30'][i] == 'Funcionarios públicos (Hugo López-Gatell, Jorge Alcocer, etc)':
    df['p_30'][i] = 2
  elif df['p_30'][i] == 'Instituciones académicas afines a la UNAM':
    df['p_30'][i] = 3
  elif df['p_30'][i] == 'Medios de comunicación nacionales (El Universal, MILENIO, Reforma, etc.)':
    df['p_30'][i] = 4
  elif df['p_30'][i] == 'Medios de comunicación internacionales (The Guardian, The Washington Post, RT)':
    df['p_30'][i] = 5
  elif df['p_30'][i] == 'Expertos de la salud (médicos, psicólogos, biólogos, epidemiólogos,etc)':
    df['p_30'][i] = 6
  elif df['p_30'][i] == 'Medios de noticias independientes (El Pulso de la República, Sin Censura Media, etc)':
    df['p_30'][i] = 7
  elif df['p_30'][i] == 'Celebridades o influencers':
    df['p_30'][i] = 8
  elif df['p_30'][i] == 'No uso o no tengo redes sociales':
    df['p_30'][i] = 9

In [ ]:
df.head(100)

,index,p_-1,p_0,p_1,p_2,p_3,p_4,p_5,p_6,p_7,p_8,p_9,p_10,p_11,p_12,p_13,p_14,p_15,p_16,p_17,p_18,p_19,p_20,p_21,p_22,p_23,p_24,p_25,p_26,p_27,p_28,p_29,p_30
0,0,2020/11/26 1:55:14 p. m. GMT-6,Sí,0,0,4,0,1,0,0,Dolor de cabeza,0,0,0,0,0,0,0,0,0,0,"Ir a un mercado, una tienda de alimentos o far...","Taxi, UBER, DiDi;Vehículo propio o prestado (a...",Pasar tiempo frecuentemente con alguien que no...,1,KN95;Mascarilla con válvula,0,2,2,1,"En pláticas con familiares, conocidos o amigos...",Facebook;Twitter;YouTube,Organismos internacionales;Instituciones acadé...
1,1,2020/11/26 1:55:40 p. m. GMT-6,Sí,1,0,4,0,2,0,1,No he tenido síntomas,0,4,0,0,0,0,0,0,0,0,"Ir a un mercado, una tienda de alimentos o far...","Taxi, UBER, DiDi;Vehículo propio o prestado (a...",Ninguna de las anteriores,1,Desechable;Tela;Quirúrgico,0,2,2,2,"Por redes sociales (Facebook, Twitter, Instagr...",Facebook;YouTube,"Gobierno Federal, Secretaría de Salud, IMSS o ..."
2,2,2020/11/26 1:55:47 p. m. GMT-6,Sí,1,0,14,1,1,1,1,Congestión nasal,0,0,0,0,0,0,0,0,0,0,Ninguna de las anteriores,"Metro, Tren suburbano o Tren ligero;Taxi, UBER...",Ninguna de las anteriores,1,Desechable;Neopreno;Quirúrgico,0,2,2,0,"Por medios tradicionales (TV, Radio, Periódico...",Facebook;Twitter,"Gobierno Federal, Secretaría de Salud, IMSS o ..."
3,3,2020/11/26 1:56:50 p. m. GMT-6,Sí,0,0,14,0,2,0,1,Pérdida del sentido del gusto o el olfato,0,4,0,0,0,0,0,0,0,0,Ninguna de las anteriores,"Bicicleta, patines, patineta u otro",Ninguna de las anteriores,1,Neopreno,1,2,2,0,"En pláticas con familiares, conocidos o amigos...",Facebook;Instagram;YouTube,Instituciones académicas afines a la UNAM
4,4,2020/11/26 1:57:01 p. m. GMT-6,Sí,1,0,14,0,1,0,0,No he tenido síntomas,0,0,0,0,0,0,0,0,0,0,"Ir a un mercado, una tienda de alimentos o far...","Metro, Tren suburbano o Tren ligero;Vehículo p...",Ninguna de las anteriores,1,Quirúrgico;KN95,0,2,2,0,"En pláticas con familiares, conocidos o amigos...",Twitter,"Gobierno Federal, Secretaría de Salud, IMSS o ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,2020/11/26 2:29:54 p. m. GMT-6,Sí,0,1,4,0,1,0,1,Tos,0,0,0,0,0,0,0,0,0,0,"Ir a un restaurante, cafetería o centro comerc...","Metro, Tren suburbano o Tren ligero;Metrobus, ...",Pasar tiempo frecuentemente con alguien que no...,1,Tela,0,2,2,1,"Por redes sociales (Facebook, Twitter, Instagr...",Facebook;Twitter,"Funcionarios públicos (Hugo López-Gatell, Jorg..."
96,97,2020/11/26 2:29:55 p. m. GMT-6,Sí,1,0,14,1,2,1,1,No he tenido síntomas,0,4,0,0,0,0,0,0,0,0,Ninguna de las anteriores,"Vehículo propio o prestado (auto, motocicleta)",Ninguna de las anteriores,1,KN95,0,2,2,1,"Por medios tradicionales (TV, Radio, Periódico...",Facebook;Instagram,Instituciones académicas afines a la UNAM;Medi...
97,98,2020/11/26 2:30:03 p. m. GMT-6,Sí,1,0,4,0,1,0,0,Dolor de cabeza,0,0,0,0,0,1,1,0,0,0,"Ir a un mercado, una tienda de alimentos o far...","Vehículo propio o prestado (auto, motocicleta)",Pasar tiempo frecuentemente con alguien que no...,1,Quirúrgico;KN95,0,2,2,2,"En pláticas con familiares, conocidos o amigos...",Twitter,"Gobierno Federal, Secretaría de Salud, IMSS o ..."
98,99,2020/11/26 2:30:08 p. m. GMT-6,Sí,0,0,4,0,1,0,1,No he tenido síntomas,1,0,1,1,1,0,0,1,3,3,"Ir a un mercado, una tienda de alimentos o far...","Metro, Tren suburbano o Tren ligero;Metrobus, ...","Ir a un hospital como voluntario, paciente o v...",1,Quirúrgico;KN95,0,2,2,0,"En pláticas con familiares, conocidos o amigos...",Facebook,"Gobierno Federal, Secretaría de Salud, IMSS o ..."


In [ ]:
df.to_csv('categorized.csv')

In [38]:
X = df.copy().loc[:,'p_1':].astype(str)
X.fillna(0, inplace=True)
X = X.values
X.shape

(997, 30)

In [41]:
oe = OrdinalEncoder()
oe.fit(X)
X_enc = oe.transform (X)
#X_test_enc = oe.transform (X_test)
X_enc

array([[  0.,   0.,   2., ...,  11.,  13., 176.],
       [  1.,   0.,   2., ...,  27.,  14.,  97.],
       [  1.,   0.,   4., ...,  26.,   7.,  97.],
       ...,
       [  0.,   0.,   2., ...,  11.,   4., 145.],
       [  0.,   0.,   2., ...,  27.,  14., 125.],
       [  1.,   1.,   2., ...,   9.,   0., 125.]])

In [47]:
oe.categories_

[array(['Hombre', 'Mujer', 'Otro', 'Prefiero no responder'], dtype=object),
 array(['Entre 18 a 24 años', 'Entre 25 años a 34 años',
        'Entre 45 años a 54 años', 'Entre 55 años a 64 años',
        'Entre 65 años a 74 años'], dtype=object),
 array(['Baja California Norte', 'Chiapas', 'Ciudad de México', 'Durango',
        'Estado de México', 'Guanajuato', 'Guerrero', 'Hidalgo', 'Jalisco',
        'Michoacán de Ocampo', 'Morelos', 'Oaxaca', 'Puebla', 'Querétaro',
        'Quintana Roo', 'San Luis Potosí', 'Tabasco', 'Tamaulipas',
        'Tlaxcala', 'Veracruz', 'Yucatán'], dtype=object),
 array(['No', 'Sí'], dtype=object),
 array(['Campo', 'Ciudad', 'Pueblo', 'Zona Rural'], dtype=object),
 array(['Ciudad', 'Pueblo', 'Zona Rural', 'nan'], dtype=object),
 array(['No', 'No sé', 'Sí'], dtype=object),
 array(['Congestión nasal', 'Congestión nasal;Dificultad para respirar',
        'Congestión nasal;Dificultad para respirar;Dolor de cabeza',
        'Congestión nasal;Dificultad para resp

In [51]:
oe.categories_[11]

array(['0', '1', '10', '12', '15', '16', '2', '20', '20 ', '25', '3',
       '3 personas de mi familia', '4', '40', '5', '6', '7', '8', '9',
       'Aproximadamente 2 o 3', 'Diez',
       'Tos y dificultad para respirar. ', 'como 12 aprox', 'nan'],
      dtype=object)

In [67]:
np.where(df['p_12']=='nan')
#df['p_12'][44] = 3
#df['p_12'][218] = 20
#df['p_12'][542] = 3
#df['p_12'][372] = 10
#df['p_12'][14] = 1
#df['p_12'][234] = 12

(array([], dtype=int64),)